Cell 1 — Install

In [ ]:
!pip install httpx selectolax pydantic tenacity aiolimiter

Cell 2 — Imports

In [ ]:
import asyncio
import random
import re
import time
from urllib.parse import urljoin

import httpx
from aiolimiter import AsyncLimiter
from pydantic import BaseModel, Field
from selectolax.parser import HTMLParser
from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

Cell 3 — Configuration
├── BACKEND_URL
├── API_PREFIX
└── PRODUCT_ENDPOINT

In [ ]:
# =========================
# Configuration
# =========================

BACKEND_URL = "http://localhost:8001"
API_PREFIX = "/api/v1"

PRODUCT_ENDPOINT = f"{BACKEND_URL}{API_PREFIX}/product"

Cell 4 — Product models

In [ ]:
from typing import Any


class ProductImage(BaseModel):
    url: str
    alt: str = ""
    is_primary: bool = False
    sort_order: int = 0


class RawProduct(BaseModel):
    source: str
    source_url: str

    external_id: str | None = None

    name: str
    brand: str | None = None

    sku: str | None = None
    barcode: str | None = None
    mpn: str | None = None
    gtin: str | None = None

    price: float | None = None
    compare_at_price: float | None = None
    currency: str = "BDT"

    availability: str | None = None

    description: str = ""
    short_description: str = ""

    images: list[ProductImage] = Field(
        default_factory=list
    )

    specifications: dict[str, Any] = Field(
        default_factory=dict
    )

    attributes: dict[str, Any] = Field(
        default_factory=dict
    )

Cell 5 — HTTP client
Rate limiting + randomized delay + 429/Retry-After handling

In [ ]:
class EcommerceHttpClient:

    def __init__(
        self,
        requests_per_minute=20,
        min_delay=2,
        max_delay=4,
        max_retries=3,
    ):
        self.limiter = AsyncLimiter(
            requests_per_minute,
            60,
        )

        self.min_delay = min_delay
        self.max_delay = max_delay
        self.last_request = 0
        self.max_retries = max_retries

        self.client = httpx.AsyncClient(
            timeout=30,
            follow_redirects=True,
            headers={
                "User-Agent": "EcommerceImporter/1.0 (product data importer)"
            },
        )

    async def get(self, url):

        for attempt in range(1, self.max_retries + 1):

            response = await self._get_once(url)

            if response.status_code != 429:
                break

            if attempt == self.max_retries:
                break

            retry_after = response.headers.get("Retry-After")

            try:
                wait = float(retry_after) if retry_after else 60
            except ValueError:
                wait = 60

            print(
                f"429 received — waiting {wait}s "
                f"before retry {attempt + 1}/{self.max_retries}"
            )

            await asyncio.sleep(wait)

        response.raise_for_status()

        return response

    async def _get_once(self, url):

        async with self.limiter:

            elapsed = (
                time.monotonic()
                - self.last_request
            )

            delay = random.uniform(
                self.min_delay,
                self.max_delay,
            )

            if elapsed < delay:
                await asyncio.sleep(
                    delay - elapsed
                )

            response = await self.client.get(url)

            self.last_request = time.monotonic()

            return response

    async def close(self):
        await self.client.aclose()

Cell 6 — Star Tech scraper

In [ ]:
class StarTechScraper:

    BASE_URL = "https://www.startech.com.bd"

    def __init__(self, client):
        self.client = client

    async def scrape_category(self, url):

        response = await self.client.get(url)

        tree = HTMLParser(response.text)

        products = []

        for item in tree.css(".p-item"):

            product = self.parse_product_card(item)

            if product:
                products.append(product)

        return products

    def parse_product_card(self, item):

        name_node = item.css_first(".p-item-name")

        if not name_node:
            return None

        link = name_node.css_first("a")

        if not link:
            return None

        name = name_node.text(strip=True)

        href = link.attributes.get("href")

        if not href:
            return None

        url = urljoin(
            self.BASE_URL,
            href,
        )

        # --- price ---
        price = None

        price_node = item.css_first(
            "meta[itemprop='price']"
        )

        if price_node:

            value = price_node.attributes.get(
                "content"
            )

            if value:
                try:
                    price = float(value)
                except ValueError:
                    pass

        # --- availability ---
        availability = None

        availability_node = item.css_first(
            "meta[itemprop='availability']"
        )

        if availability_node:

            availability = availability_node.attributes.get(
                "content"
            )

            if availability:
                availability = availability.split("/")[-1]

        # --- image ---
        images = []

        image_node = item.css_first(
            "img[itemprop='image']"
        )

        if image_node:

            image_url = (
                image_node.attributes.get("src")
                or image_node.attributes.get("data-src")
            )

            if image_url:

                images.append(
                    ProductImage(
                        url=urljoin(
                            self.BASE_URL,
                            image_url,
                        ),
                        alt=name,
                        is_primary=True,
                    )
                )

        # --- specifications from the short-description list ---
        specifications = {}

        for li in item.css(".short-description li"):

            text = li.text(strip=True)

            if ":" not in text:
                continue

            key, value = text.split(":", 1)

            specifications[
                key.strip()
            ] = value.strip()

        # --- external id from the add-to-cart button ---
        external_id = None

        onclick_node = item.css_first(
            "[onclick*='cart.add']"
        )

        if onclick_node:

            onclick = onclick_node.attributes.get(
                "onclick",
                "",
            )

            match = re.search(
                r"cart\.add\(['\"]([^'\"]+)",
                onclick,
            )

            if match:
                external_id = match.group(1)

        return RawProduct(
            source="startech",
            source_url=url,
            external_id=external_id,
            name=name,
            price=price,
            currency="BDT",
            availability=availability,
            images=images,
            specifications=specifications,
        )

Cell 7 — Backend sender
Tenacity: 3 attempts, exponential backoff (1–10s), reraises the last error

In [ ]:
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type((httpx.HTTPError,)),
    reraise=True,
)
async def send_product_to_backend(product, http_client):

    payload = product.model_dump()

    response = await http_client.client.post(
        PRODUCT_ENDPOINT,
        json=payload,
    )

    response.raise_for_status()

    return response.json()

Cell 8 — Test — scrape one category page

In [ ]:
http_client = EcommerceHttpClient()

scraper = StarTechScraper(http_client)

products = await scraper.scrape_category(
    "https://www.startech.com.bd/air-conditioner"
)

print(f"Scraped {len(products)} products")

if products:
    print(products[0].model_dump_json(indent=2))

Cell 9 — Test — scrape multiple pages

In [ ]:
async def scrape_category_pages(
    scraper,
    base_url,
    max_pages=5,
):
    all_products = []

    for page in range(1, max_pages + 1):

        url = (
            base_url
            if page == 1
            else f"{base_url}?page={page}"
        )

        print(f"Scraping page {page}: {url}")

        page_products = await scraper.scrape_category(url)

        if not page_products:
            break

        all_products.extend(page_products)

    return all_products


products = await scrape_category_pages(
    scraper,
    "https://www.startech.com.bd/air-conditioner",
    max_pages=3,
)

print(f"Total products: {len(products)}")

Cell 10 — Test — send a product to the backend, then close the client

In [ ]:
if products:
    result = await send_product_to_backend(products[0], http_client)

    print(result)
else:
    print("No products to send — nothing posted to the backend")

# Done — release the underlying connection pool
await http_client.close()